In [1]:
from opt_targeted_transfers import BinaryRateTargetedTransfers
from opt_targeted_transfers import Dataset, split
from data_loaders import load_data, PATH_TO_TRAIN_DATA, PATH_TO_TEST_DATA

In [2]:
# Make train and test set
train_data = load_data(PATH_TO_TRAIN_DATA)
test_data = load_data(PATH_TO_TEST_DATA)

train_dataset = Dataset(df=train_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])
train_dataset, validation_dataset = split(train_dataset)
test_covariate_dataset = Dataset(df=test_data, outcome=None, weight='hh_wgt', covs=['hh_size', 'urban'])
test_dataset = Dataset(df=test_data, outcome='consumption_per_capita_per_day', weight='hh_wgt', covs=['hh_size', 'urban'])

In [3]:
tt = BinaryRateTargetedTransfers(c_bar=2.15, n_regressors=5)

In [4]:
# Nuisance parameter estimation
# Fit conditional improvement regressors for different transfer values
tt.fit(train_dataset, validation_dataset)

Fitting conditional binary_rate improvement for transfer size 0.01


100%|██████████| 300/300 [00:03<00:00, 84.27it/s, val loss=0.171] 


Fitting conditional binary_rate improvement for transfer size 0.545


100%|██████████| 300/300 [00:03<00:00, 76.90it/s, val loss=1.08]


Fitting conditional binary_rate improvement for transfer size 1.08


100%|██████████| 300/300 [00:04<00:00, 72.87it/s, val loss=1.01] 


Fitting conditional binary_rate improvement for transfer size 1.615


100%|██████████| 300/300 [00:04<00:00, 73.12it/s, val loss=0.896]


Fitting conditional binary_rate improvement for transfer size 2.15


100%|██████████| 300/300 [00:03<00:00, 87.69it/s, val loss=0.888]


In [5]:
# Precomputation for policy optimization step.
import numpy as np
budgets = np.linspace(0.05, 2.15, 10)
tt.get_opt_transfer_sizes_given_budget_grid(validation_dataset, budgets=budgets)

In [6]:
# Set budget and run policy optimization step for that budget.
# Policy optimization step returns transfer amount for each unit in the test set.
# Note that budget must lie in the set of budgets used in the precomputation step.
tt.set_budget(budget=budgets[2])
assignments = tt.run_opt(test_covariate_dataset)

In [7]:
# Evaluate policy. 
res = tt.evaluate(test_dataset)
res

{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.18212575272768974,
 'post_transfer_poverty_rate': 0.39140739451531265,
 'policy_cost_per_capita': 0.5165927711236471,
 'budget': 0.5166666666666667,
 'policy_type': 'binary_rate',
 'd': 2}

In [8]:
# Can try a different budget without redoing the fit step and precomputation step.
# Note that budget must lie in the set of budgets used in the precomputation step.
# Setting the budget will clear assignments attribute.
tt.set_budget(budgets[-2])
tt.run_opt(test_covariate_dataset)
res = tt.evaluate(test_dataset)
res


{'initial_poverty_rate': 0.5774688939291257,
 'initial_poverty_gap': 0.44336819493851437,
 'post_transfer_poverty_gap': 0.013310025987358125,
 'post_transfer_poverty_rate': 0.022186748789001395,
 'policy_cost_per_capita': 1.91621885014867,
 'budget': 1.9166666666666667,
 'policy_type': 'binary_rate',
 'd': 2}

In [9]:
tt.compute_auc(test_covariate_dataset=test_covariate_dataset, 
               test_dataset=test_dataset, 
               metrics=['post_transfer_poverty_rate', 'post_transfer_poverty_gap'], 
               budgets=budgets)

{'post_transfer_poverty_rate': {'auc': 0.4694120940502852,
  'results': [0.5736151502248663,
   0.5736151502248663,
   0.39140739451531265,
   0.3840218578623006,
   0.16489764710898905,
   0.14865127807484912,
   0.03029695490464681,
   0.009881510765965823,
   0.022186748789001395,
   0.0]},
 'post_transfer_poverty_gap': {'auc': 0.2627099991593136,
  'results': [0.437608388543916,
   0.437608388543916,
   0.18212575272768974,
   0.17748544655685067,
   0.046504606734439846,
   0.03657205121046011,
   0.012771399231883128,
   0.0007181311325025541,
   0.013310025987358125,
   0.0]}}